# Clasificador Jerárquico de Patologías Estructurales

Enfoque en dos niveles para mejorar la clasificación de 96 defectos:

- **Nivel 1:** Predecir la categoría (11 clases)
- **Nivel 2:** Predecir el defecto específico dentro de cada categoría

Comparación final contra el modelo plano de 96 clases.

In [ ]:
# ============================================================
# Celda 1: Importar librerías
# ============================================================

import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
print("Librerías cargadas.")

In [ ]:
# ============================================================
# Celda 2: Cargar dataset y catálogo JSON
# El CSV tiene 2216 observaciones con 20 features y defecto_numero.
# Del JSON extraemos el mapeo defecto_numero → categoría para
# crear la columna target del Nivel 1.
# ============================================================

df = pd.read_csv("dataset_patologias_sintetico.csv")

with open("patologias_estructurales.json", encoding="utf-8") as f:
    catalogo = json.load(f)

# Diccionario: número → info completa de la patología
catalogo_dict = {p["numero"]: p for p in catalogo}

# Mapeo: número de defecto → categoría del JSON
mapa_categoria = {p["numero"]: p["categoria"] for p in catalogo}

# Agregar columna "categoria" al dataset
df["categoria"] = df["defecto_numero"].map(mapa_categoria)

print(f"Dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nCategorías ({df['categoria'].nunique()}):")
print(df["categoria"].value_counts().to_string())

In [ ]:
# ============================================================
# Celda 3: Preparar features y targets
# X = las 20 features de observación (todas categóricas).
# y_cat = categoría (para Nivel 1, 11 clases).
# y_def = defecto_numero (para Nivel 2 y modelo plano).
# ============================================================

columnas_features = [c for c in df.columns if c not in ["defecto_numero", "categoria"]]

X = df[columnas_features]
y_cat = df["categoria"]                    # Target Nivel 1
y_def = df["defecto_numero"]               # Target Nivel 2

# Preprocesador reutilizable para todos los modelos
preprocesador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_features)
    ]
)

print(f"Features: {len(columnas_features)}")
print(f"Target Nivel 1: {y_cat.nunique()} categorías")
print(f"Target Nivel 2: {y_def.nunique()} defectos")

In [ ]:
# ============================================================
# Celda 4: Split global 80/20
# Usamos el MISMO split para todos los modelos, estratificando
# por defecto_numero para mantener la proporción de todas las
# 96 clases (y por ende las 11 categorías) en train y test.
# ============================================================

X_train, X_test, y_cat_train, y_cat_test, y_def_train, y_def_test = (
    train_test_split(X, y_cat, y_def, test_size=0.2, stratify=y_def, random_state=42)
)

# Reconstruir DataFrames con ambos targets para filtrar después
train_df = X_train.copy()
train_df["categoria"] = y_cat_train.values
train_df["defecto_numero"] = y_def_train.values

test_df = X_test.copy()
test_df["categoria"] = y_cat_test.values
test_df["defecto_numero"] = y_def_test.values

print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")
print(f"Categorías en train: {y_cat_train.nunique()} | en test: {y_cat_test.nunique()}")

## Nivel 1: Modelo de Categoría (11 clases)

Predice a qué familia pertenece el defecto: Pilares, Vigas, Ménsulas, Viguetas, Voladizos, Forjados, Deformaciones, Térmicos, Cerramientos, Cimentaciones, Pilares y Vigas.

In [ ]:
# ============================================================
# Celda 5: Entrenar modelo Nivel 1 (categoría)
# LabelEncoder convierte las 11 categorías a enteros 0-10
# porque XGBoost necesita targets numéricos.
# Usamos Pipeline = ColumnTransformer + XGBClassifier.
# ============================================================

le_cat = LabelEncoder()
y_cat_train_enc = le_cat.fit_transform(y_cat_train)
y_cat_test_enc = le_cat.transform(y_cat_test)

modelo_nivel1 = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("clasificador", XGBClassifier(
        n_estimators=200, eta=0.1, max_depth=6, gamma=0.5,
        objective="multi:softprob", eval_metric="mlogloss",
        random_state=42, verbosity=0
    ))
])

modelo_nivel1.fit(X_train, y_cat_train_enc)

# Evaluar sobreajuste
acc_n1_train = accuracy_score(y_cat_train_enc, modelo_nivel1.predict(X_train))
acc_n1_test = accuracy_score(y_cat_test_enc, modelo_nivel1.predict(X_test))

print(f"NIVEL 1 — Modelo de Categoría (11 clases)")
print(f"  Accuracy TRAIN: {acc_n1_train:.4f}")
print(f"  Accuracy TEST:  {acc_n1_test:.4f}")
print(f"  Sobreajuste:    {acc_n1_train - acc_n1_test:.4f}")

In [ ]:
# ============================================================
# Celda 6: Classification report y confusion matrix del Nivel 1
# Con 11 clases la matriz de confusión es legible y muestra
# dónde se confunden las categorías entre sí.
# ============================================================

y_pred_n1 = modelo_nivel1.predict(X_test)
nombres_categorias = le_cat.classes_

print("CLASSIFICATION REPORT — NIVEL 1:")
print(classification_report(y_cat_test_enc, y_pred_n1,
                            target_names=nombres_categorias, zero_division=0))

fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_predictions(
    y_cat_test_enc, y_pred_n1,
    display_labels=nombres_categorias,
    cmap="Blues", xticks_rotation=45, ax=ax
)
ax.set_title("Matriz de Confusión — Nivel 1 (Categorías)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrix_nivel1.png", dpi=150, bbox_inches="tight")
plt.show()

## Nivel 2: Modelos por Categoría

Para cada una de las 11 categorías se entrena un modelo especializado que solo clasifica entre los defectos de esa familia. Esto reduce drásticamente el número de clases por modelo (de 96 a entre 4 y 16).

In [ ]:
# ============================================================
# Celda 7: Entrenar modelos de Nivel 2 (uno por categoría)
# Para cada categoría:
#   1. Filtrar solo las filas de esa categoría en train y test.
#   2. Crear un LabelEncoder local (defectos → 0,1,2...).
#   3. Entrenar un Pipeline (preprocesador + XGBClassifier).
#   4. Evaluar accuracy en train y test.
# Se guarda todo en un diccionario para uso posterior.
# ============================================================

modelos_nivel2 = {}   # {categoria: {modelo, le, acc_train, acc_test, n_clases}}
categorias = sorted(y_cat.unique())

print(f"{'Categoría':<30} {'Clases':>6} {'Train':>7} {'Acc Train':>10} {'Acc Test':>10}")
print("─" * 75)

for cat in categorias:
    # Filtrar filas de esta categoría
    mask_train = train_df["categoria"] == cat
    mask_test = test_df["categoria"] == cat

    X_tr = train_df.loc[mask_train, columnas_features]
    y_tr = train_df.loc[mask_train, "defecto_numero"]
    X_te = test_df.loc[mask_test, columnas_features]
    y_te = test_df.loc[mask_test, "defecto_numero"]

    # Si no hay muestras en test, saltar (no debería pasar con stratify)
    if len(X_te) == 0:
        print(f"  {cat:<30} — SIN DATOS EN TEST, OMITIDO")
        continue

    # LabelEncoder local para esta categoría
    le_local = LabelEncoder()
    y_tr_enc = le_local.fit_transform(y_tr)
    y_te_enc = le_local.transform(y_te)
    n_clases = len(le_local.classes_)

    # Pipeline con preprocesador nuevo (clon del original)
    prep_local = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_features)
        ]
    )

    modelo_n2 = Pipeline(steps=[
        ("preprocesador", prep_local),
        ("clasificador", XGBClassifier(
            n_estimators=200, eta=0.1, max_depth=5, gamma=0.5,
            objective="multi:softprob" if n_clases > 2 else "binary:logistic",
            eval_metric="mlogloss" if n_clases > 2 else "logloss",
            random_state=42, verbosity=0
        ))
    ])

    modelo_n2.fit(X_tr, y_tr_enc)

    acc_tr = accuracy_score(y_tr_enc, modelo_n2.predict(X_tr))
    acc_te = accuracy_score(y_te_enc, modelo_n2.predict(X_te))

    modelos_nivel2[cat] = {
        "modelo": modelo_n2,
        "le": le_local,
        "acc_train": acc_tr,
        "acc_test": acc_te,
        "n_clases": n_clases,
    }

    print(f"  {cat:<30} {n_clases:>4}   {len(X_tr):>5}   {acc_tr:>9.4f}   {acc_te:>9.4f}")

# Accuracy promedio ponderado de Nivel 2
total_test = sum(len(test_df[test_df["categoria"] == c]) for c in modelos_nivel2)
acc_n2_promedio = sum(
    info["acc_test"] * len(test_df[test_df["categoria"] == c])
    for c, info in modelos_nivel2.items()
) / total_test

print(f"\n{'─' * 75}")
print(f"  Accuracy promedio ponderado Nivel 2: {acc_n2_promedio:.4f}")

## Diagnóstico jerárquico encadenado

La observación pasa primero por el Nivel 1 (categoría) y después por el Nivel 2 (defecto específico dentro de esa categoría). Se muestran probabilidades de ambos niveles, top 3 defectos, causas y medidas.

In [ ]:
# ============================================================
# Celda 8: Función de diagnóstico jerárquico
# Paso 1: predict_proba en Nivel 1 → categoría más probable.
# Paso 2: predict_proba en Nivel 2 de esa categoría → top 3
#          defectos específicos con causas y medidas del JSON.
# ============================================================

def diagnostico_jerarquico(observacion, top_n=3):
    """Diagnóstico en dos niveles: categoría → defecto específico."""
    obs_df = pd.DataFrame([observacion])

    # ── Nivel 1: predecir categoría ──
    probas_n1 = modelo_nivel1.predict_proba(obs_df)[0]
    idx_n1 = np.argmax(probas_n1)
    categoria_pred = le_cat.inverse_transform([idx_n1])[0]
    confianza_n1 = probas_n1[idx_n1] * 100

    print("=" * 70)
    print("OBSERVACIÓN DEL INSPECTOR:")
    print("=" * 70)
    for campo, valor in observacion.items():
        print(f"  {campo:<25} → {valor}")

    print(f"\n{'─' * 70}")
    print(f"  NIVEL 1 — CATEGORÍA PREDICHA:")
    print(f"  [{confianza_n1:.1f}%] {categoria_pred}")

    # ── Nivel 2: predecir defecto específico ──
    if categoria_pred not in modelos_nivel2:
        print(f"\n  ⚠ No hay modelo de Nivel 2 para '{categoria_pred}'")
        print(f"{'=' * 70}\n")
        return categoria_pred, None

    info_n2 = modelos_nivel2[categoria_pred]
    modelo_n2 = info_n2["modelo"]
    le_n2 = info_n2["le"]

    probas_n2 = modelo_n2.predict_proba(obs_df)[0]
    top_indices_n2 = np.argsort(probas_n2)[::-1][:top_n]

    print(f"\n{'─' * 70}")
    print(f"  NIVEL 2 — TOP {top_n} DEFECTOS EN '{categoria_pred}':")
    print(f"{'─' * 70}")

    defecto_top1 = None
    for rank, idx in enumerate(top_indices_n2, 1):
        num_defecto = le_n2.inverse_transform([idx])[0]  # número real 1-96
        confianza_n2 = probas_n2[idx] * 100
        patologia = catalogo_dict.get(num_defecto)

        if rank == 1:
            defecto_top1 = num_defecto

        if patologia:
            print(f"\n  {rank}. [{confianza_n2:.1f}%] Defecto #{num_defecto}: {patologia['defecto']}")
            print(f"     Gravedad: {patologia['gravedad']}")
            print(f"     Causas probables:")
            for causa in patologia["causas"]:
                print(f"       • {causa}")
            print(f"     Medidas de precaución:")
            for medida in patologia["medidas"]:
                print(f"       ✓ {medida}")

    print(f"\n{'=' * 70}\n")
    return categoria_pred, defecto_top1

print("Función diagnostico_jerarquico() definida.")

## Simulación de campo

Los mismos 4 ejemplos del notebook anterior para comparar directamente qué predecía el modelo plano (96 clases) vs el jerárquico (2 niveles).

In [ ]:
# ============================================================
# Celda 9: Definir los 4 ejemplos de campo
# Son los mismos del notebook clasificador_patologias_xgboost.ipynb
# para poder comparar resultados lado a lado.
# ============================================================

# Ejemplo 1: Pilar con fisura diagonal tras sismo → esperado: Cortante en pilar (#3)
ejemplo_1 = {
    "tipo_elemento": "pilar", "orientacion_fisura": "diagonal_45_75",
    "abertura": "abierta", "ubicacion_en_elemento": "ambas_caras",
    "patron": "unica", "comportamiento": "aparecio_de_golpe",
    "desprendimiento": "recubrimiento_desprendido", "color": "normal",
    "textura": "normal", "armadura_visible": "parcialmente",
    "corrosion": "no_visible", "estado_estribos": "separados",
    "edad_estructura": "5_a_20", "ambiente": "exterior_continental",
    "velocidad_aparicion": "subita_reciente", "carga": "impacto_sismo",
    "intervencion_previa": "original", "deformacion_visible": "desplazamiento_lateral",
    "sonido_golpe": "diferente_zonas", "presencia_agua": "seco",
}

# Ejemplo 2: Viga con fisuras verticales centro y flecha → esperado: Flexión en viga (#15)
ejemplo_2 = {
    "tipo_elemento": "viga", "orientacion_fisura": "vertical",
    "abertura": "media", "ubicacion_en_elemento": "centro",
    "patron": "multiples_paralelas", "comportamiento": "se_abre_con_tiempo",
    "desprendimiento": "no_hay", "color": "normal",
    "textura": "normal", "armadura_visible": "no",
    "corrosion": "no_visible", "estado_estribos": "correctos",
    "edad_estructura": "5_a_20", "ambiente": "interior",
    "velocidad_aparicion": "anos_despues", "carga": "sobrecarga",
    "intervencion_previa": "original", "deformacion_visible": "flecha_abajo",
    "sonido_golpe": "solido", "presencia_agua": "seco",
}

# Ejemplo 3: Vigueta costera con óxido → esperado: Corrosión en viguetas (#43)
ejemplo_3 = {
    "tipo_elemento": "vigueta", "orientacion_fisura": "horizontal",
    "abertura": "cerrada_fina", "ubicacion_en_elemento": "cara_inferior",
    "patron": "multiples_paralelas", "comportamiento": "se_abre_con_tiempo",
    "desprendimiento": "recubrimiento_desprendido", "color": "manchas_oxido",
    "textura": "humedo_filtraciones", "armadura_visible": "parcialmente",
    "corrosion": "oxidada_visible", "estado_estribos": "correctos",
    "edad_estructura": "mas_50", "ambiente": "costero_marino",
    "velocidad_aparicion": "anos_despues", "carga": "normal",
    "intervencion_previa": "original", "deformacion_visible": "no",
    "sonido_golpe": "hueco", "presencia_agua": "salpicadura",
}

# Ejemplo 4: Zapata con grieta diagonal y giro → esperado: Asiento de zapata (#86)
ejemplo_4 = {
    "tipo_elemento": "cimentacion_zapata", "orientacion_fisura": "diagonal_45",
    "abertura": "abierta", "ubicacion_en_elemento": "union_otro_elemento",
    "patron": "unica", "comportamiento": "se_abre_con_tiempo",
    "desprendimiento": "no_hay", "color": "normal",
    "textura": "humedo_filtraciones", "armadura_visible": "no",
    "corrosion": "no_visible", "estado_estribos": "correctos",
    "edad_estructura": "20_a_50", "ambiente": "exterior_continental",
    "velocidad_aparicion": "anos_despues", "carga": "normal",
    "intervencion_previa": "original", "deformacion_visible": "giro_inclinacion",
    "sonido_golpe": "solido", "presencia_agua": "humedo",
}

ejemplos = [
    ("Pilar + sismo + diagonal (esperado: #3 Cortante en pilar)", ejemplo_1),
    ("Viga + fisuras verticales + flecha (esperado: #15 Flexión en viga)", ejemplo_2),
    ("Vigueta costera + óxido (esperado: #43 Corrosión en viguetas)", ejemplo_3),
    ("Zapata + grieta diagonal + giro (esperado: #86 Asiento de zapata)", ejemplo_4),
]

print(f"4 ejemplos de campo definidos.")

In [ ]:
# ============================================================
# Celda 10: Ejecutar diagnóstico jerárquico en los 4 ejemplos
# ============================================================

for descripcion, obs in ejemplos:
    print(f"\n{'▶':} {descripcion}")
    diagnostico_jerarquico(obs)

## Comparación: Modelo plano vs Jerárquico

Entrenamos un modelo plano de 96 clases con el mismo split y comparamos accuracy lado a lado.

In [ ]:
# ============================================================
# Celda 11: Entrenar modelo plano (96 clases) como referencia
# Mismo split, mismos hiperparámetros, para comparar limpiamente.
# ============================================================

le_plano = LabelEncoder()
y_def_train_enc = le_plano.fit_transform(y_def_train)
y_def_test_enc = le_plano.transform(y_def_test)

prep_plano = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_features)
    ]
)

modelo_plano = Pipeline(steps=[
    ("preprocesador", prep_plano),
    ("clasificador", XGBClassifier(
        n_estimators=200, eta=0.1, max_depth=6, gamma=0.5,
        objective="multi:softprob", eval_metric="mlogloss",
        random_state=42, verbosity=0
    ))
])

modelo_plano.fit(X_train, y_def_train_enc)

acc_plano_train = accuracy_score(y_def_train_enc, modelo_plano.predict(X_train))
acc_plano_test = accuracy_score(y_def_test_enc, modelo_plano.predict(X_test))

print(f"Modelo plano (96 clases) → Train: {acc_plano_train:.4f} | Test: {acc_plano_test:.4f}")

In [ ]:
# ============================================================
# Celda 12: Accuracy end-to-end del clasificador jerárquico
# Para medir la accuracy real del sistema completo, simulamos
# el encadenamiento: Nivel 1 predice categoría → Nivel 2
# predice defecto. Si el Nivel 1 falla, el Nivel 2 también.
# Esto da la accuracy real comparada con el modelo plano.
# ============================================================

aciertos_jerarquico = 0
total = len(X_test)

for i in range(total):
    obs = X_test.iloc[[i]]

    # Nivel 1: predecir categoría
    idx_n1 = modelo_nivel1.predict(obs)[0]
    cat_pred = le_cat.inverse_transform([idx_n1])[0]

    # Nivel 2: predecir defecto dentro de la categoría predicha
    if cat_pred in modelos_nivel2:
        info = modelos_nivel2[cat_pred]
        idx_n2 = info["modelo"].predict(obs)[0]
        defecto_pred = info["le"].inverse_transform([idx_n2])[0]
    else:
        defecto_pred = -1  # fallback

    if defecto_pred == y_def_test.iloc[i]:
        aciertos_jerarquico += 1

acc_jerarquico_e2e = aciertos_jerarquico / total

print(f"Accuracy end-to-end del jerárquico: {acc_jerarquico_e2e:.4f}")

In [ ]:
# ============================================================
# Celda 13: Tabla comparativa final
# Muestra lado a lado: modelo plano vs cada nivel jerárquico
# vs la accuracy real end-to-end del sistema encadenado.
# ============================================================

tabla = pd.DataFrame({
    "Modelo": [
        "Plano (96 clases)",
        "Jerárquico Nivel 1 (11 categorías)",
        "Jerárquico Nivel 2 (prom. ponderado)",
        "Jerárquico end-to-end (encadenado)",
    ],
    "Acc Train": [
        acc_plano_train,
        acc_n1_train,
        np.mean([info["acc_train"] for info in modelos_nivel2.values()]),
        np.nan,  # no aplica
    ],
    "Acc Test": [
        acc_plano_test,
        acc_n1_test,
        acc_n2_promedio,
        acc_jerarquico_e2e,
    ],
})

# Comparar el modelo plano con las predicciones del jerárquico en los 4 ejemplos
print("=" * 70)
print("COMPARACIÓN DE PREDICCIONES EN LOS 4 EJEMPLOS")
print("=" * 70)
print(f"\n{'Ejemplo':<12} {'Esperado':<35} {'Plano (96)':<20} {'Jerárquico':<20}")
print("─" * 90)

for i, (desc, obs) in enumerate(ejemplos, 1):
    esperado = desc.split("esperado: ")[1].rstrip(")")

    # Modelo plano
    obs_df = pd.DataFrame([obs])
    probas_plano = modelo_plano.predict_proba(obs_df)[0]
    idx_plano = np.argmax(probas_plano)
    def_plano = le_plano.inverse_transform([idx_plano])[0]
    conf_plano = probas_plano[idx_plano] * 100
    nombre_plano = catalogo_dict[def_plano]["defecto"][:25]

    # Modelo jerárquico
    idx_n1 = modelo_nivel1.predict(obs_df)[0]
    cat_pred = le_cat.inverse_transform([idx_n1])[0]
    if cat_pred in modelos_nivel2:
        info = modelos_nivel2[cat_pred]
        probas_n2 = info["modelo"].predict_proba(obs_df)[0]
        idx_n2 = np.argmax(probas_n2)
        def_jer = info["le"].inverse_transform([idx_n2])[0]
        conf_jer = probas_n2[idx_n2] * 100
        nombre_jer = catalogo_dict[def_jer]["defecto"][:25]
    else:
        def_jer, conf_jer, nombre_jer = "?", 0, "?"

    print(f"  Ej. {i:<6} {esperado:<35} #{def_plano} ({conf_plano:.0f}%){'':>3} #{def_jer} ({conf_jer:.0f}%)")

print(f"\n{'=' * 70}")
print("\nTABLA RESUMEN DE ACCURACY:")
print("=" * 70)
print(tabla.to_string(index=False, float_format="{:.4f}".format))
print("=" * 70)

# Calcular mejora
mejora = acc_jerarquico_e2e - acc_plano_test
signo = "+" if mejora > 0 else ""
print(f"\nMejora del jerárquico vs plano: {signo}{mejora:.4f} ({signo}{mejora*100:.1f} puntos porcentuales)")

---

## Optimización del Clasificador Jerárquico

Tres estrategias para mejorar el rendimiento sin cambiar la estructura ni los datos:

1. **Feature engineering:** combinaciones de features que capturan interacciones clave.
2. **Pesos por gravedad:** `sample_weight` para penalizar errores en patologías graves.
3. **Regularización con GridSearchCV:** búsqueda de hiperparámetros que reduzcan sobreajuste.

In [ ]:
# ============================================================
# Celda 14: Feature engineering — features combinadas
# Creamos 3 columnas nuevas que capturan interacciones entre
# features que ayudan a diferenciar defectos similares.
# Por ejemplo: "pilar_diagonal_45" es muy distinto de
# "viga_diagonal_45" para distinguir cortante en pilar vs viga.
# ============================================================

def agregar_features_combinadas(dataframe):
    """Agrega features de interacción al DataFrame (modifica in-place)."""
    df_out = dataframe.copy()
    df_out["elemento_orientacion"] = df_out["tipo_elemento"] + "_" + df_out["orientacion_fisura"]
    df_out["elemento_ubicacion"] = df_out["tipo_elemento"] + "_" + df_out["ubicacion_en_elemento"]
    df_out["contexto_ambiente"] = df_out["edad_estructura"] + "_" + df_out["ambiente"]
    return df_out

# Aplicar al dataset completo
df_opt = agregar_features_combinadas(df)

# Nuevas columnas de features (las originales + las 3 nuevas)
columnas_features_opt = columnas_features + [
    "elemento_orientacion", "elemento_ubicacion", "contexto_ambiente"
]

print(f"Features originales: {len(columnas_features)}")
print(f"Features con combinadas: {len(columnas_features_opt)}")
print(f"\nEjemplos de valores nuevos:")
print(f"  elemento_orientacion: {df_opt['elemento_orientacion'].unique()[:5]}")
print(f"  elemento_ubicacion:   {df_opt['elemento_ubicacion'].unique()[:5]}")
print(f"  contexto_ambiente:    {df_opt['contexto_ambiente'].unique()[:5]}")

In [ ]:
# ============================================================
# Celda 15: Crear pesos por gravedad (sample_weight)
# Las patologías más graves (gravedad_puntos=4) reciben peso 4,
# las de 3 peso 3, etc. Así el modelo penaliza más los errores
# en defectos peligrosos como aplastamiento o punzonamiento.
# ============================================================

mapa_gravedad = {p["numero"]: p["gravedad_puntos"] for p in catalogo}
df_opt["gravedad_peso"] = df_opt["defecto_numero"].map(mapa_gravedad)

print("Distribución de pesos por gravedad:")
print(df_opt["gravedad_peso"].value_counts().sort_index().to_string())
print(f"\nPeso medio: {df_opt['gravedad_peso'].mean():.2f}")

In [ ]:
# ============================================================
# Celda 16: Split optimizado con features combinadas y pesos
# Mismo split 80/20 con random_state=42 y stratify por defecto.
# Ahora incluye las 3 features nuevas y la columna de pesos.
# ============================================================

X_opt = df_opt[columnas_features_opt]
y_cat_opt = df_opt["categoria"]
y_def_opt = df_opt["defecto_numero"]
pesos_opt = df_opt["gravedad_peso"]

(X_opt_train, X_opt_test,
 y_cat_opt_train, y_cat_opt_test,
 y_def_opt_train, y_def_opt_test,
 pesos_train, pesos_test) = train_test_split(
    X_opt, y_cat_opt, y_def_opt, pesos_opt,
    test_size=0.2, stratify=y_def_opt, random_state=42
)

# DataFrames auxiliares para filtrar por categoría en Nivel 2
train_opt_df = X_opt_train.copy()
train_opt_df["categoria"] = y_cat_opt_train.values
train_opt_df["defecto_numero"] = y_def_opt_train.values
train_opt_df["peso"] = pesos_train.values

test_opt_df = X_opt_test.copy()
test_opt_df["categoria"] = y_cat_opt_test.values
test_opt_df["defecto_numero"] = y_def_opt_test.values
test_opt_df["peso"] = pesos_test.values

# Preprocesador con las columnas ampliadas
preprocesador_opt = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_features_opt)
    ]
)

print(f"Train: {X_opt_train.shape[0]} | Test: {X_opt_test.shape[0]}")
print(f"Features: {X_opt_train.shape[1]} (20 originales + 3 combinadas)")

### GridSearchCV — Nivel 1 optimizado

Búsqueda de hiperparámetros con regularización fuerte para reducir sobreajuste.  
Se pasan `sample_weight` al fit para penalizar errores en patologías graves.

In [ ]:
# ============================================================
# Celda 17: GridSearchCV para Nivel 1 optimizado
# Grilla con parámetros de regularización para reducir la
# brecha train-test. Se usa cv=5 y refit por f1_macro.
# sample_weight se pasa al clasificador dentro del Pipeline
# usando la sintaxis "clasificador__sample_weight".
# ============================================================

from sklearn.model_selection import GridSearchCV

le_cat_opt = LabelEncoder()
y_cat_opt_train_enc = le_cat_opt.fit_transform(y_cat_opt_train)
y_cat_opt_test_enc = le_cat_opt.transform(y_cat_opt_test)

pipeline_n1_opt = Pipeline(steps=[
    ("preprocesador", preprocesador_opt),
    ("clasificador", XGBClassifier(
        objective="multi:softprob", eval_metric="mlogloss",
        random_state=42, verbosity=0
    ))
])

grilla_n1 = {
    "clasificador__n_estimators": [100, 200],
    "clasificador__eta": [0.05, 0.1],
    "clasificador__max_depth": [2, 3, 4],
    "clasificador__min_child_weight": [3, 5, 10],
    "clasificador__gamma": [1, 3, 5],
    "clasificador__reg_lambda": [1, 5, 10],
    "clasificador__reg_alpha": [0, 0.5, 1],
}

n_combos = 1
for v in grilla_n1.values():
    n_combos *= len(v)
print(f"Nivel 1: {n_combos} combinaciones × 5 folds = {n_combos * 5} fits")
print("Esto puede tardar varios minutos...\n")

grid_n1 = GridSearchCV(
    estimator=pipeline_n1_opt,
    param_grid=grilla_n1,
    scoring="f1_macro",
    refit=True,
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

# Pasar sample_weight al clasificador dentro del Pipeline
grid_n1.fit(X_opt_train, y_cat_opt_train_enc,
            clasificador__sample_weight=pesos_train.values)

print(f"\nMejor f1_macro CV: {grid_n1.best_score_:.4f}")
print(f"Mejores hiperparámetros:")
for p, v in grid_n1.best_params_.items():
    print(f"  {p}: {v}")

In [ ]:
# ============================================================
# Celda 18: Evaluar Nivel 1 optimizado — sobreajuste
# Comparamos la brecha train-test antes vs después de optimizar.
# ============================================================

modelo_n1_opt = grid_n1.best_estimator_

acc_n1_opt_train = accuracy_score(y_cat_opt_train_enc, modelo_n1_opt.predict(X_opt_train))
acc_n1_opt_test = accuracy_score(y_cat_opt_test_enc, modelo_n1_opt.predict(X_opt_test))

print("NIVEL 1 — COMPARACIÓN DE SOBREAJUSTE:")
print("=" * 55)
print(f"{'Métrica':<25} {'Original':>12} {'Optimizado':>12}")
print("─" * 55)
print(f"{'Accuracy Train':<25} {acc_n1_train:>11.4f} {acc_n1_opt_train:>12.4f}")
print(f"{'Accuracy Test':<25} {acc_n1_test:>11.4f} {acc_n1_opt_test:>12.4f}")
print(f"{'Brecha (sobreajuste)':<25} {acc_n1_train - acc_n1_test:>11.4f} {acc_n1_opt_train - acc_n1_opt_test:>12.4f}")
print(f"{'f1_macro CV':<25} {'—':>12} {grid_n1.best_score_:>12.4f}")

### GridSearchCV — Nivel 2 optimizado (por categoría)

Para cada categoría, GridSearchCV con la misma grilla de regularización, `sample_weight` por gravedad, y features combinadas.

In [ ]:
# ============================================================
# Celda 19: GridSearchCV para cada modelo de Nivel 2
# Para categorías con pocas clases (4) usamos una grilla más
# reducida para evitar tiempos excesivos. Se pasan sample_weight.
# ============================================================

modelos_n2_opt = {}

# Grilla reducida para categorías con pocas muestras
grilla_n2 = {
    "clasificador__n_estimators": [100, 200],
    "clasificador__eta": [0.05, 0.1],
    "clasificador__max_depth": [2, 3, 4],
    "clasificador__min_child_weight": [3, 5, 10],
    "clasificador__gamma": [1, 3, 5],
    "clasificador__reg_lambda": [1, 5, 10],
    "clasificador__reg_alpha": [0, 0.5, 1],
}

print(f"{'Categoría':<30} {'Clases':>6} {'Acc Train':>10} {'Acc Test':>10} {'f1 CV':>8} {'Brecha':>8}")
print("─" * 80)

for cat in categorias:
    mask_train = train_opt_df["categoria"] == cat
    mask_test = test_opt_df["categoria"] == cat

    X_tr = train_opt_df.loc[mask_train, columnas_features_opt]
    y_tr = train_opt_df.loc[mask_train, "defecto_numero"]
    w_tr = train_opt_df.loc[mask_train, "peso"]
    X_te = test_opt_df.loc[mask_test, columnas_features_opt]
    y_te = test_opt_df.loc[mask_test, "defecto_numero"]

    if len(X_te) == 0:
        continue

    le_local = LabelEncoder()
    y_tr_enc = le_local.fit_transform(y_tr)
    y_te_enc = le_local.transform(y_te)
    n_clases = len(le_local.classes_)

    # cv no puede ser mayor que la clase más pequeña
    min_muestras_clase = pd.Series(y_tr_enc).value_counts().min()
    cv_folds = min(5, min_muestras_clase)
    if cv_folds < 2:
        cv_folds = 2

    prep_local = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_features_opt)
        ]
    )

    pipe_n2 = Pipeline(steps=[
        ("preprocesador", prep_local),
        ("clasificador", XGBClassifier(
            objective="multi:softprob" if n_clases > 2 else "binary:logistic",
            eval_metric="mlogloss" if n_clases > 2 else "logloss",
            random_state=42, verbosity=0
        ))
    ])

    grid_n2 = GridSearchCV(
        estimator=pipe_n2,
        param_grid=grilla_n2,
        scoring="f1_macro",
        refit=True,
        cv=cv_folds,
        n_jobs=-1,
        verbose=0,
        return_train_score=True,
    )

    grid_n2.fit(X_tr, y_tr_enc, clasificador__sample_weight=w_tr.values)

    mejor_n2 = grid_n2.best_estimator_
    acc_tr = accuracy_score(y_tr_enc, mejor_n2.predict(X_tr))
    acc_te = accuracy_score(y_te_enc, mejor_n2.predict(X_te))

    modelos_n2_opt[cat] = {
        "modelo": mejor_n2,
        "le": le_local,
        "acc_train": acc_tr,
        "acc_test": acc_te,
        "n_clases": n_clases,
        "best_score": grid_n2.best_score_,
    }

    brecha = acc_tr - acc_te
    print(f"  {cat:<30} {n_clases:>4}   {acc_tr:>9.4f}   {acc_te:>9.4f}  {grid_n2.best_score_:>6.4f}  {brecha:>7.4f}")

# Accuracy promedio ponderado
total_test_opt = sum(len(test_opt_df[test_opt_df["categoria"] == c]) for c in modelos_n2_opt)
acc_n2_opt_promedio = sum(
    info["acc_test"] * len(test_opt_df[test_opt_df["categoria"] == c])
    for c, info in modelos_n2_opt.items()
) / total_test_opt

print(f"\n{'─' * 80}")
print(f"  Accuracy promedio ponderado N2 optimizado: {acc_n2_opt_promedio:.4f} (antes: {acc_n2_promedio:.4f})")

In [ ]:
# ============================================================
# Celda 20: Accuracy end-to-end del jerárquico optimizado
# y recall en patologías con gravedad_puntos=4
# ============================================================

from sklearn.metrics import f1_score, recall_score

# ── End-to-end optimizado ──
aciertos_opt = 0
y_pred_opt_e2e = []

for i in range(len(X_opt_test)):
    obs = X_opt_test.iloc[[i]]
    idx_n1 = modelo_n1_opt.predict(obs)[0]
    cat_pred = le_cat_opt.inverse_transform([idx_n1])[0]
    if cat_pred in modelos_n2_opt:
        info = modelos_n2_opt[cat_pred]
        idx_n2 = info["modelo"].predict(obs)[0]
        def_pred = info["le"].inverse_transform([idx_n2])[0]
    else:
        def_pred = -1
    y_pred_opt_e2e.append(def_pred)
    if def_pred == y_def_opt_test.iloc[i]:
        aciertos_opt += 1

acc_opt_e2e = aciertos_opt / len(X_opt_test)
y_pred_opt_e2e = np.array(y_pred_opt_e2e)
y_true_test = y_def_opt_test.values

# ── f1_macro end-to-end ──
# Necesitamos manejar clases predichas como -1 (fallback)
clases_validas = sorted(set(y_true_test) | set(y_pred_opt_e2e[y_pred_opt_e2e > 0]))
f1_opt_e2e = f1_score(y_true_test, y_pred_opt_e2e, average="macro", zero_division=0)

# ── Recall en gravedad 4 ──
# Filtrar solo las muestras cuyo defecto real tiene gravedad_puntos=4
mask_grav4 = np.array([mapa_gravedad.get(d, 0) == 4 for d in y_true_test])
n_grav4 = mask_grav4.sum()

# Recall gravedad 4 del jerárquico original (sin optimizar)
y_pred_orig_e2e = []
for i in range(len(X_test)):
    obs = X_test.iloc[[i]]
    idx_n1 = modelo_nivel1.predict(obs)[0]
    cat_pred = le_cat.inverse_transform([idx_n1])[0]
    if cat_pred in modelos_nivel2:
        info = modelos_nivel2[cat_pred]
        idx_n2 = info["modelo"].predict(obs)[0]
        def_pred = info["le"].inverse_transform([idx_n2])[0]
    else:
        def_pred = -1
    y_pred_orig_e2e.append(def_pred)
y_pred_orig_e2e = np.array(y_pred_orig_e2e)

# Recall gravedad 4: proporción de aciertos en muestras graves
recall_g4_orig = (y_pred_orig_e2e[mask_grav4] == y_true_test[mask_grav4]).mean()
recall_g4_opt = (y_pred_opt_e2e[mask_grav4] == y_true_test[mask_grav4]).mean()

# Recall gravedad 4 del modelo plano
y_pred_plano_all = le_plano.inverse_transform(modelo_plano.predict(X_test))
recall_g4_plano = (y_pred_plano_all[mask_grav4] == y_true_test[mask_grav4]).mean()

# f1_macro de las versiones anteriores
f1_plano = f1_score(y_true_test, y_pred_plano_all, average="macro", zero_division=0)
f1_orig_e2e = f1_score(y_true_test, y_pred_orig_e2e, average="macro", zero_division=0)

print(f"Muestras con gravedad 4 en test: {n_grav4} de {len(y_true_test)}")
print(f"\nRecall en gravedad 4:")
print(f"  Plano original:         {recall_g4_plano:.4f}")
print(f"  Jerárquico original:    {recall_g4_orig:.4f}")
print(f"  Jerárquico optimizado:  {recall_g4_opt:.4f}")

### Comparación final: 3 versiones lado a lado

In [ ]:
# ============================================================
# Celda 21: Tabla comparativa de las 3 versiones
# Modelo plano (96), jerárquico original, jerárquico optimizado.
# ============================================================

tabla_final = pd.DataFrame({
    "Modelo": [
        "Plano (96 clases)",
        "Jerárquico original",
        "Jerárquico optimizado",
    ],
    "Acc Test": [
        acc_plano_test,
        acc_jerarquico_e2e,
        acc_opt_e2e,
    ],
    "f1_macro Test": [
        f1_plano,
        f1_orig_e2e,
        f1_opt_e2e,
    ],
    "Brecha Train-Test N1": [
        "—",
        f"{acc_n1_train - acc_n1_test:.4f}",
        f"{acc_n1_opt_train - acc_n1_opt_test:.4f}",
    ],
    "Recall Gravedad 4": [
        recall_g4_plano,
        recall_g4_orig,
        recall_g4_opt,
    ],
})

print("=" * 85)
print("TABLA COMPARATIVA — 3 VERSIONES DEL CLASIFICADOR")
print("=" * 85)
print(tabla_final.to_string(index=False))
print("=" * 85)

# Interpretación
mejora_acc = acc_opt_e2e - acc_plano_test
mejora_f1 = f1_opt_e2e - f1_plano
mejora_g4 = recall_g4_opt - recall_g4_plano
print(f"\nMejora del optimizado vs plano:")
print(f"  Accuracy:          {mejora_acc:+.4f} ({mejora_acc*100:+.1f} pp)")
print(f"  f1_macro:          {mejora_f1:+.4f} ({mejora_f1*100:+.1f} pp)")
print(f"  Recall gravedad 4: {mejora_g4:+.4f} ({mejora_g4*100:+.1f} pp)")

In [ ]:
# ============================================================
# Celda 22: Simulación de campo — 4 ejemplos × 3 versiones
# Para cada ejemplo mostramos qué predice el modelo plano,
# el jerárquico original y el jerárquico optimizado.
# ============================================================

def predecir_plano(obs):
    """Predicción del modelo plano (96 clases)."""
    obs_df = pd.DataFrame([obs])
    probas = modelo_plano.predict_proba(obs_df)[0]
    idx = np.argmax(probas)
    num = le_plano.inverse_transform([idx])[0]
    return num, probas[idx] * 100

def predecir_jerarquico_orig(obs):
    """Predicción del jerárquico original."""
    obs_df = pd.DataFrame([obs])
    idx_n1 = modelo_nivel1.predict(obs_df)[0]
    cat = le_cat.inverse_transform([idx_n1])[0]
    if cat in modelos_nivel2:
        info = modelos_nivel2[cat]
        probas = info["modelo"].predict_proba(obs_df)[0]
        idx_n2 = np.argmax(probas)
        num = info["le"].inverse_transform([idx_n2])[0]
        return num, probas[idx_n2] * 100, cat
    return -1, 0, cat

def predecir_jerarquico_opt(obs):
    """Predicción del jerárquico optimizado (features combinadas)."""
    obs_ext = obs.copy()
    obs_ext["elemento_orientacion"] = obs["tipo_elemento"] + "_" + obs["orientacion_fisura"]
    obs_ext["elemento_ubicacion"] = obs["tipo_elemento"] + "_" + obs["ubicacion_en_elemento"]
    obs_ext["contexto_ambiente"] = obs["edad_estructura"] + "_" + obs["ambiente"]
    obs_df = pd.DataFrame([obs_ext])
    idx_n1 = modelo_n1_opt.predict(obs_df)[0]
    cat = le_cat_opt.inverse_transform([idx_n1])[0]
    if cat in modelos_n2_opt:
        info = modelos_n2_opt[cat]
        probas = info["modelo"].predict_proba(obs_df)[0]
        idx_n2 = np.argmax(probas)
        num = info["le"].inverse_transform([idx_n2])[0]
        return num, probas[idx_n2] * 100, cat
    return -1, 0, cat

print("=" * 95)
print("COMPARACIÓN DE PREDICCIONES — 4 EJEMPLOS × 3 VERSIONES")
print("=" * 95)
print(f"\n{'Ej':<4} {'Esperado':<25} {'Plano':<22} {'Jerárq. orig.':<22} {'Jerárq. optim.':<22}")
print("─" * 95)

for i, (desc, obs) in enumerate(ejemplos, 1):
    esperado = desc.split("esperado: ")[1].rstrip(")")

    num_p, conf_p = predecir_plano(obs)
    num_o, conf_o, cat_o = predecir_jerarquico_orig(obs)
    num_opt, conf_opt, cat_opt = predecir_jerarquico_opt(obs)

    col_p = f"#{num_p} ({conf_p:.0f}%)"
    col_o = f"#{num_o} ({conf_o:.0f}%)"
    col_opt = f"#{num_opt} ({conf_opt:.0f}%)"

    print(f"  {i:<3} {esperado:<25} {col_p:<22} {col_o:<22} {col_opt:<22}")

print("─" * 95)

# Detalle del jerárquico optimizado para cada ejemplo
print(f"\n{'=' * 95}")
print("DETALLE DEL JERÁRQUICO OPTIMIZADO:")
print("=" * 95)

for i, (desc, obs) in enumerate(ejemplos, 1):
    obs_ext = obs.copy()
    obs_ext["elemento_orientacion"] = obs["tipo_elemento"] + "_" + obs["orientacion_fisura"]
    obs_ext["elemento_ubicacion"] = obs["tipo_elemento"] + "_" + obs["ubicacion_en_elemento"]
    obs_ext["contexto_ambiente"] = obs["edad_estructura"] + "_" + obs["ambiente"]
    obs_df = pd.DataFrame([obs_ext])

    # Nivel 1
    probas_n1 = modelo_n1_opt.predict_proba(obs_df)[0]
    idx_n1 = np.argmax(probas_n1)
    cat = le_cat_opt.inverse_transform([idx_n1])[0]
    conf_n1 = probas_n1[idx_n1] * 100

    print(f"\n  Ejemplo {i}: N1 → [{conf_n1:.1f}%] {cat}")

    if cat in modelos_n2_opt:
        info = modelos_n2_opt[cat]
        probas_n2 = info["modelo"].predict_proba(obs_df)[0]
        top3 = np.argsort(probas_n2)[::-1][:3]
        for rank, idx in enumerate(top3, 1):
            num = info["le"].inverse_transform([idx])[0]
            conf = probas_n2[idx] * 100
            nombre = catalogo_dict[num]["defecto"]
            grav = catalogo_dict[num]["gravedad"]
            print(f"           N2 #{rank}: [{conf:.1f}%] Defecto #{num}: {nombre} (Grav: {grav})")
            if rank == 1:
                print(f"           Causas: {'; '.join(catalogo_dict[num]['causas'][:2])}")
                print(f"           Medidas: {'; '.join(catalogo_dict[num]['medidas'][:2])}")

print(f"\n{'=' * 95}")